In [1]:
import sys
import json
import random
from pathlib import Path

# Make the repository root importable from notebooks
root_candidates = []
if "__file__" in globals():
    root_candidates.append(Path(__file__).resolve().parent.parent)
root_candidates.extend([Path.cwd().resolve(), Path.cwd().resolve().parent])

for candidate in root_candidates:
    if (candidate / "retrieval").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

from retrieval.pipeline import retrieve
from generation.citation_generator import generate_answer
from agents.validation_agent import validation_node

RANDOM_SEED = 42
random.seed(RANDOM_SEED)


In [ ]:
new_training_queries = [
    "What is an object in object-oriented programming?",
    "What is the purpose of a constructor in C++?",
    "What is the this pointer in C++?",
    "What is inheritance in object-oriented programming?",
    "What is the difference between a public and a private member in a C++ class?",
    "Why did George Murchison say Beneatha's natural hair was \"eccentric\"?",
    "What is an adjective with examples?",
    "What is the main theme of the poem \"Harlem\" by Langston Hughes that serves as the epigraph for A Raisin in the Sun?",
    "What is the role of the friend function in C++?",
    "What is the difference between a function template and a class template in C++?",
    "Can the array class template be used with user-defined types?",
    "What is the purpose of a virtual destructor in C++?",
    "What is the difference between virtual functions and pure virtual functions?",
    "What is Lennie's primary motivation for wanting to tend the rabbits?",
    "Where does the story of mice and men take place?",
    "What is the purpose of the getline() function in C++?",
    "What is the primary advantage of using NumPy arrays over Python lists?",
    "What are stream manipulators and how can they be used to format output?",
    "What is the core difference between a try and catch block in C++?",
    "What does a virtual function allow in C++?",
    "What is a lambda function in Python?",
    "What happens to characters that do not match the expected type when using C's cin >> operator?",
    "How does stringtokenizer work in Java?",
    "اردو اور ہندی میں کیا فرق ہے؟",
    "اردو کا قومی زبان ہونے کا کیا مطلب ہے؟",
    "نیکوسیا کا منقسم ہونے کا کیا مطلب ہے؟",
    "ہندی اور اردو کی ادبی روایات میں کیا فرق ہے؟",
    "اردو کی شناخت میں فارسی-عربی رسم الخط کا کیا کردار ہے؟",
    "1930년 FIFA 월드컵에서 우승한 국가는 어디이며, 결승전 점수는 어떻게 되었나요?",
    "대한민국 국회의원의 임기는 몇 년이며, 불체포특권은 어떤 경우에 적용되나요?",
    "2026년 베네수엘라 지진의 지질학적 원인은 무엇입니까?",
    "미국 보수 기독교계에서는 기독교 근본주의를 주장하였다",
    "미국 보수 기독교계의 근본주의가 한국 개신교에 어떤 영향을 미쳤는가?",
]
print(f"{len(new_training_queries)} new queries loaded")

In [2]:
with open(r"C:\Users\zinsi\Downloads\training_needs_generation_v3.json", encoding="utf-8") as f:
    new_training_queries = json.load(f)
print(f"{len(new_training_queries)} queries loaded")

27 queries loaded


In [ ]:
from retrieval.bootstrap import PROJ2_SRC
EVAL_QUERIES_PATH = PROJ2_SRC.parent / "eval" / "eval_queries.json"

with open(EVAL_QUERIES_PATH, encoding="utf-8") as f:
    all_labeled = json.load(f)["queries"]

print(f"{len(all_labeled)} labeled queries loaded")

In [ ]:
# stratified split: 30 -> training pool, 15 -> held-out eval (by language)
from collections import defaultdict

by_lang = defaultdict(list)
for q in all_labeled:
    by_lang[q["language"]].append(q)

EVAL_COUNTS = {"en": 7, "ur": 4, "ko": 4}  # 15 total held out
eval_set = []
training_labeled = []

for lang, queries in by_lang.items():
    shuffled = queries[:]
    random.shuffle(shuffled)
    n_eval = EVAL_COUNTS.get(lang, 0)
    eval_set.extend(shuffled[:n_eval])
    training_labeled.extend(shuffled[n_eval:])

print(f"Eval set: {len(eval_set)} queries")
print(f"Training pool (labeled): {len(training_labeled)} queries")

# save the eval set now — this is untouched by anything below
Path("finetuning").mkdir(exist_ok=True)
with open("finetuning/eval_set.json", "w", encoding="utf-8") as f:
    json.dump(eval_set, f, ensure_ascii=False, indent=2)

In [3]:
training_query_pool = list(new_training_queries)
random.shuffle(training_query_pool)
print(f"Total training query pool: {len(training_query_pool)} unique queries")

Total training query pool: 27 unique queries


In [ ]:
# build the full training query pool: labeled (query text only, chunk_id
# not used for training) + new queries
training_query_pool = [q["query"] for q in training_labeled] + new_training_queries
random.shuffle(training_query_pool)

print(f"Total training query pool: {len(training_query_pool)} unique queries")

In [4]:
# generate candidate training triples, with checkpointing.
# No security/validation LLM calls — these are known-good queries, so we
# use free heuristic checks instead: retrieval confidence gate (unchanged)
# + citation-marker presence + not a hedge response. One generation per
# query, not three, since there's no validator picking the best of several.

import re

#CHECKPOINT_PATH = Path("finetuning/training_checkpoint.json")
CHECKPOINT_PATH = Path("finetuning/training_checkpoint_new.json")  # separate from the original run
HEDGE_PHRASES = ["doesn't clearly address", "not confident enough", "does not contain information"]


def looks_like_good_training_example(answer: str) -> bool:
    if any(phrase in answer.lower() for phrase in HEDGE_PHRASES):
        return False
    if not re.search(r'\[\d+\]', answer):  # no citation markers at all
        return False
    return True


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_checkpoint(checkpoint):
    CHECKPOINT_PATH.parent.mkdir(exist_ok=True)
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(checkpoint, f, ensure_ascii=False, indent=2)


checkpoint = load_checkpoint()
print(f"Resuming — {len(checkpoint)}/{len(training_query_pool)} queries already processed")

for i, query in enumerate(training_query_pool, 1):
    if query in checkpoint:
        print(f"[{i}/{len(training_query_pool)}] SKIP (already done): {query[:60]!r}")
        continue

    print(f"[{i}/{len(training_query_pool)}] {query[:60]!r}")

    try:
        chunks = retrieve(query, top_k=5, candidate_pool=10, history=[])
        if not chunks or chunks[0][2] < 0.25:
            print("  skipped — low retrieval confidence")
            checkpoint[query] = {"example": None, "status": "skipped_low_confidence"}
            save_checkpoint(checkpoint)
            continue

        result = generate_answer(query, chunks)

        if looks_like_good_training_example(result["answer"]):
            example = {
                "query": query,
                "context_chunks": [
                    {"chunk_id": cid, "text": chunk["text"], "source_doc": chunk.get("source_doc", cid)}
                    for cid, chunk, score in chunks
                ],
                "answer": result["answer"],
                "sources": result["sources"],
            }
            checkpoint[query] = {"example": example, "status": "done"}
            print("  kept ✓")
        else:
            checkpoint[query] = {"example": None, "status": "hedge_or_no_citation"}
            print("  skipped — hedge or no citations")

        save_checkpoint(checkpoint)

    except Exception as e:
        print(f"  ERROR: {e}")
        checkpoint[query] = {"example": None, "status": "error", "error": str(e)}
        save_checkpoint(checkpoint)

print(f"\nDone. {sum(1 for v in checkpoint.values() if v['example'])} examples kept out of {len(checkpoint)} queries.")

Resuming — 0/27 queries already processed
[1/27] 'How does a goal-based agent decide what action to take?'
[bootstrap] loading persisted FAISS index...
[DEBUG] raw='How does a goal-based agent decide what action to take?' -> rewritten='What decision-making process does a goal-based agent use to select its next action.'
[bm25] loading persisted index...
[vector_search] loading BAAI/bge-m3...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 27206.82it/s]


[rerank] loading BAAI/bge-reranker-base...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 1137.41it/s]


  kept ✓
[2/27] 'What is the syntax for a lambda function in Python?'
[DEBUG] raw='What is the syntax for a lambda function in Python?' -> rewritten='Syntax of a lambda function in Python'
  kept ✓
[3/27] "What does the witches' prophecy say to Macbeth?"
[DEBUG] raw="What does the witches' prophecy say to Macbeth?" -> rewritten="What is the content of the witches' prophecy revealed to Macbeth?"
  skipped — hedge or no citations
[4/27] 'Who invented the goal-based agent architecture and in what y'
[DEBUG] raw='Who invented the goal-based agent architecture and in what year?' -> rewritten='What is the origin of the goal-based agent architecture and when was it developed'
  skipped — low retrieval confidence
[5/27] 'Why do genomic datasets skew toward European ancestry?'
[DEBUG] raw='Why do genomic datasets skew toward European ancestry?' -> rewritten='What factors contribute to the overrepresentation of European ancestry in genomic datasets?'
  kept ✓
[6/27] '구석기 시대 사람들은 어떤 도구를 사용했나요?'
[

In [ ]:
from llm.client import MODELS
print(MODELS)

In [ ]:
# retry error / hedge-or-no-citation / low-confidence queries
retry_statuses = {"error", "hedge_or_no_citation", "skipped_low_confidence"}
to_retry = [q for q, d in checkpoint.items() if d["status"] in retry_statuses]

print(f"{len(to_retry)} queries to retry:")
'''
for q in to_retry:
    print(f"  [{checkpoint[q]['status']}] {q[:60]!r}")

for q in to_retry:
    del checkpoint[q]
save_checkpoint(checkpoint)

print("\nCleared — rerun Cell 6 to regenerate these")
'''

In [6]:
# flatten checkpoint into final training set, dedupe, save
all_examples = [v["example"] for v in checkpoint.values() if v.get("example")]

seen = set()
deduped = []
for ex in all_examples:
    key = (ex["query"], ex["answer"])
    if key not in seen:
        seen.add(key)
        deduped.append(ex)

print(f"{len(deduped)} unique examples after dedup")

status_counts = {}
for data in checkpoint.values():
    status_counts[data["status"]] = status_counts.get(data["status"], 0) + 1
print(f"Query status breakdown: {status_counts}")

#with open("finetuning/training_examples.json", "w", encoding="utf-8") as f:
with open("finetuning/training_examples_v3.json", "w", encoding="utf-8") as f:
    json.dump(deduped, f, ensure_ascii=False, indent=2)
print("Saved")

17 unique examples after dedup
Query status breakdown: {'done': 17, 'hedge_or_no_citation': 4, 'skipped_low_confidence': 6}
Saved
